# Custom wake-word trainer — bulletproof Colab notebook (2026 edition)

Train the FALKOR wake word model with openWakeWord in Google Colab. **Run all → walk away → download `.onnx`** at the end.

## Why this notebook exists

openWakeWord's official Colab notebook bit-rotted hard against Python 3.12 (no piper-phonemize wheels), against torchaudio 2.x (`set_audio_backend` removed), against newer Colab images (HF Hub timeouts, namespace-package edge cases), and against its own internal config schema (4 keys silently moved to required-or-`KeyError` over the past year).

This notebook is patched against all of that. Every cell is **self-healing** — checks its own outputs, re-creates what's missing. There's a hard pre-flight gate (cell 4) that catches every dep/file issue before any of the slow downloads start.

It also **replaces openwakeword's `auto_train` with a faithful hand-rolled PyTorch loop** that mirrors auto_train's curriculum exactly: 3-stage learning rate (1e-4 → 1e-5 → 1e-6), negative-weight ramp 1→1500, hard-negative mining, FP-per-hour validation against an ACAV100M continuous-audio slice, 90/90/10 percentile checkpoint ensemble averaging. The upstream `auto_train` path keeps surfacing bugs against `mmap_batch_generator` shape handling on current openwakeword `master`; the hand-rolled trainer in cell 14 is small enough to read in one sitting and debug if anything ever changes.

## Run order

1. **Runtime → Change runtime type → L4 GPU + High RAM** (Colab Pro, $10/mo). Free T4 also works but is ~2× slower.
2. **Cell 10 — edit `TARGET_PHRASE` and `MODEL_NAME`** for your wake word. (Default = `mr graves`, the example we trained with.)
3. **Runtime → Run all**
4. Walk away ~75-90 min. Last cell downloads `<MODEL_NAME>.onnx`.

## Validated example (Mr Graves, 2026-05-09)

13 real-world utterances on a Pixel — every utterance fired:

```
WAKE — score=0.99664426
WAKE — score=0.59243464
WAKE — score=0.65439160
WAKE — score=0.77693284
WAKE — score=0.85433600
WAKE — score=0.64701974
WAKE — score=0.80747485
WAKE — score=0.95243360
WAKE — score=0.99938180
WAKE — score=0.90104705
WAKE — score=0.99639570
WAKE — score=0.98879445
WAKE — score=0.51673140
```

13/13 fires, scores 0.52-0.999, **zero phantom fires** during 30 min of normal phone use (TV, music, conversation, kitchen noise). Matches openwakeword's pretrained Hey Jarvis baseline (0.984/0.989) on clear utterances.

For deeper context on what auto_train actually does and why every part of the curriculum matters, the comments in cell 14 walk through each piece.

## 1. Comprehensive install â€” apt + pip + verify

Every dep + transitive we've discovered. `piper-tts` goes LAST via `--no-deps` so no later install can clobber it (piper-tts pulls regular `piper-phonemize` which has no Py 3.12 wheel; `piper-phonemize-cross` provides the same module name and is installed first).

In [ ]:
# Fail immediately on unsupported Colab runtimes instead of spending time downloading data.
import sys, platform
print('Python:', sys.version)
if sys.implementation.name != 'cpython' or sys.version_info < (3, 9) or sys.version_info >= (3, 13):
    raise RuntimeError(
        'Unsupported Python runtime for this notebook: ' + sys.version.split()[0] +
        '. Select a Colab runtime using CPython 3.9–3.12, then restart and run again.'
    )
if platform.system() != 'Linux' or platform.machine().lower() not in ('x86_64', 'amd64'):
    raise RuntimeError(
        'This notebook currently expects a Linux x86-64 Colab runtime; got ' +
        platform.system() + '/' + platform.machine()
    )

# Native deps
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# Install and verify Piper phonemizer on its own first. If this fails, stop here.
# Version 1.2.1 publishes CPython 3.12 Linux x86-64 wheels.
!python -m pip install --no-cache-dir "piper-phonemize-cross==1.2.1"
from piper_phonemize import phonemize_espeak
print('  Piper phonemizer import: OK')

# Python deps — piper-tts LAST via --no-deps so it cannot replace the phonemizer.
!python -m pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub
!python -m pip install -q --no-deps piper-tts

# Verify imports before any model/data downloads.
import torch, torchinfo, torchmetrics, scipy, numpy
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests
from piper import PiperVoice, SynthesisConfig
print('  All required imports (including Piper) succeeded.')


## 2. Clone repos + download model (self-healing)

Each piece (piper-sample-generator clone, libritts .pt model, openwakeword clone) is checked individually and re-created if missing or partial. Pin piper-sample-generator to commit `1a8c49bd^` â€” the parent of "Move to package", which is the last revision with flat-layout `generate_samples.py` at the root that openwakeword's train.py expects to import.

In [ ]:
import os, sys
os.chdir('/content')

# â”€â”€ piper-sample-generator (pinned to flat-layout commit) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PSG_DIR = '/content/piper-sample-generator'
PSG_PIN = '1a8c49bd29b3a132721086ee88f2253f788594a8^'
PSG_GS  = f'{PSG_DIR}/generate_samples.py'
if not os.path.exists(PSG_GS):
    print('  piper-sample-generator missing or broken â€” re-cloning')
    !rm -rf {PSG_DIR}
    !git clone -q https://github.com/rhasspy/piper-sample-generator {PSG_DIR}
!cd {PSG_DIR} && git fetch -q --all && git checkout -q {PSG_PIN}
assert os.path.exists(PSG_GS), 'piper-sample-generator pin did not yield generate_samples.py'
print(f'  âœ“ piper-sample-generator pinned: {PSG_GS} ({os.path.getsize(PSG_GS)} bytes)')

# â”€â”€ libritts model (~200 MB) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
PIPER_MODEL_URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    print(f'  libritts model missing/partial â€” downloading (~200 MB)')
    !mkdir -p {PSG_DIR}/models
    !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {PIPER_MODEL_URL}
assert os.path.getsize(PIPER_MODEL) > 100_000_000, f'libritts model only {os.path.getsize(PIPER_MODEL)} bytes â€” download failed'
print(f'  âœ“ libritts model: {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB')

# â”€â”€ openwakeword (master) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'
if not os.path.exists(OWW_TRAIN):
    print('  openwakeword missing or gutted â€” re-cloning')
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
    !pip install -q -e {OWW_DIR}
assert os.path.exists(OWW_TRAIN), 'openwakeword train.py missing after clone'
print(f'  âœ“ openwakeword: {OWW_TRAIN} ({os.path.getsize(OWW_TRAIN)/1e3:.0f} KB)')

# â”€â”€ Make sure openwakeword's PARENT dir is on sys.path so the package
# resolves correctly. /content is also on sys.path by default in Colab,
# which would resolve `import openwakeword` to /content/openwakeword/
# (the editable-install ROOT, no __init__.py) instead of the package
# at /content/openwakeword/openwakeword/. That would treat openwakeword
# as a namespace package and submodule imports (.data, .utils) would fail.
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
# Wipe any stale namespace-package state in sys.modules
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None, (
    f'openwakeword still loaded as namespace package â€” '
    f'__file__ is None, __path__={openwakeword.__path__}. '
    f'Check sys.path for stray /content entries.'
)
print(f'  âœ“ openwakeword importable: {openwakeword.__file__}')

## 3. Apply runtime patches (idempotent)

Six patches. Each is a string check + targeted modification â€” re-running this cell after a partial install reapplies cleanly without double-patching.

In [ ]:
import os, glob

# Patch A: torch_audiomentations 0.11 calls torchaudio.set_audio_backend
# which torchaudio 2.x removed. sed-replace the bad line with no-op.
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"
    print(f'  Patch A (set_audio_backend): {path}')

# Patch B: copy generate_samples.py into openwakeword's package dir.
src = '/content/piper-sample-generator/generate_samples.py'
dst = '/content/openwakeword/openwakeword/generate_samples.py'
assert os.path.exists(src), 'piper-sample-generator/generate_samples.py missing â€” cell 2 failed'
if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(src):
    !cp "{src}" "{dst}"
print(f'  Patch B (generate_samples copy): {dst}')

# Patch C: HF Hub default 10s timeouts â†’ 120s
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)
print('  Patch C (HF Hub timeouts â†’ 120s)')

# Patch D: torchaudio.info shim (removed in torchaudio 2.x)
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# â”€â”€ PATCH: info() shim for torchaudio 2.x â”€â”€'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)
print(f'  Patch D (torchaudio.info shim): {hasattr(torchaudio, "info")}')

# Patch E: openwakeword's generate_samples requires `model` arg but train.py
# doesn't pass it â€” default to the libritts model we already downloaded.
TARGET = '/content/piper-sample-generator/generate_samples.py'
!sed -i 's|model: Union\[str, Path\],|model: Union[str, Path] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",|' "{TARGET}"
!cp "{TARGET}" /content/openwakeword/openwakeword/generate_samples.py
print('  Patch E (generate_samples model arg default)')

# Patch F: train.py val dtype cast (line ~519 + ~547)
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"
print('  Patch F (train.py val dtype cast)')
print()
print('All 6 patches applied (idempotent).')

## 4. Pre-flight â€” every dep imports + every external file exists

Hard-fails if anything's wrong. This is the gate before slow downloads start. Catches new failure modes future Colab updates introduce.

In [ ]:
import os, importlib

# Files
expected = {
    'piper-sample-generator/generate_samples.py': '/content/piper-sample-generator/generate_samples.py',
    'openwakeword/generate_samples.py':            '/content/openwakeword/openwakeword/generate_samples.py',
    'libritts model':                              '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt',
    'openwakeword/train.py':                       '/content/openwakeword/openwakeword/train.py',
    'openwakeword/data.py':                        '/content/openwakeword/openwakeword/data.py',
    'openwakeword/utils.py':                       '/content/openwakeword/openwakeword/utils.py',
}
missing_files = []
for name, p in expected.items():
    if os.path.exists(p):
        size = os.path.getsize(p)
        unit = 'MB' if size > 1e6 else 'KB'
        denom = 1e6 if unit == 'MB' else 1e3
        print(f'  âœ“ {name}: {size/denom:.1f} {unit}')
    else:
        missing_files.append((name, p))
        print(f'  âœ— MISSING: {name} ({p})')

# Imports
missing_mods = []
for mod in ['torch', 'torchinfo', 'torchmetrics', 'scipy', 'numpy', 'tqdm',
            'yaml', 'mutagen', 'pronouncing', 'torchaudio', 'audiomentations',
            'torch_audiomentations', 'speechbrain', 'acoustics', 'onnx',
            'onnxruntime', 'soundfile', 'requests', 'huggingface_hub',
            'piper_phonemize', 'piper', 'openwakeword']:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing_mods.append((mod, str(e)))
        print(f'  âœ— IMPORT FAIL: {mod} ({type(e).__name__}: {e})')

# Dry-run openwakeword.data + utils â€” catches weird internal import bugs
try:
    from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
    from openwakeword.utils import compute_features_from_generator, AudioFeatures
    print('  âœ“ openwakeword.data + .utils dry-import OK')
except Exception as e:
    print(f'  âœ— openwakeword internal import: {type(e).__name__}: {e}')
    missing_mods.append(('openwakeword.data/utils', str(e)))

import torch
print(f'\n  CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')
if missing_files or missing_mods:
    raise RuntimeError(f'Pre-flight failed â€” {len(missing_files)} missing files, {len(missing_mods)} import errors. Re-run cells 1-3.')
print('\n  Pre-flight PASSED. Safe to proceed with downloads + training.')

## 5. Download openwakeword shared models (mel + embedding)

Required by augment_clips' feature-extraction step. Idempotent â€” skips files already cached.

In [ ]:
import os
models_dir = '/content/openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    out = f'{models_dir}/{fname}'
    if os.path.exists(out) and os.path.getsize(out) > 1000:
        print(f'  cached: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')
    else:
        !wget -q --tries=5 --timeout=120 "{BASE}/{fname}" -O "{out}"
        print(f'  âœ“ downloaded: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')

## 6. MIT impulse responses â†’ 16-kHz WAVs (~1 min)

Caches the HF dataset, then converts to WAV. Both halves idempotent.

In [ ]:
import os, time, numpy as np
import scipy.io.wavfile as wavfile
from huggingface_hub import snapshot_download
import datasets

output_dir = '/content/mit_rirs'
os.makedirs(output_dir, exist_ok=True)
if len([f for f in os.listdir(output_dir) if f.endswith('.wav')]) >= 250:
    print(f'  cached: {len(os.listdir(output_dir))} MIT IR WAVs')
else:
    # Pre-cache via snapshot_download (more reliable than load_dataset's internal
    # 10s timeout on 270 file metadata calls).
    for i in range(6):
        try:
            snapshot_download(repo_id='davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', etag_timeout=120, max_workers=4)
            break
        except Exception as e:
            wait = 15 * (i+1)
            print(f'  snapshot_download attempt {i+1}/6: {type(e).__name__}: {str(e)[:80]} â€” sleep {wait}s')
            time.sleep(wait)
    # Load + convert
    for attempt in range(4):
        try:
            rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                                split='train', streaming=False)
            break
        except Exception as e:
            print(f'  load_dataset attempt {attempt+1}: {type(e).__name__}: {str(e)[:120]}')
            time.sleep(20 * (attempt + 1))
    else:
        raise RuntimeError('load_dataset failed for MIT IRs')
    n = 0
    for i, row in enumerate(rir_dataset):
        out = f'{output_dir}/{i:04d}.wav'
        if os.path.exists(out): continue
        audio = row['audio']
        sr = audio['sampling_rate']
        arr = np.asarray(audio['array'])
        if sr != 16000:
            from scipy.signal import resample_poly
            arr = resample_poly(arr, 16000, sr)
        arr = (arr * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(out, 16000, arr); n += 1
    print(f'  âœ“ wrote {n} new WAVs (total {len(os.listdir(output_dir))})')

## 7. Download a compact ACAV training subset + validation data; stream background music

The notebook uses HTTP byte ranges to download only the first 10% of the official ACAV100M feature array (about 1.73 GB, representing roughly 200 hours) for real-world negative training. It separately downloads the official 185 MB false-positive validation set (about 11 hours) and streams around one hour of FMA background clips. The full 17.3 GB ACAV file and the ~8 GB FMA ZIP are not downloaded. If the host does not support byte ranges, the notebook stops rather than silently downloading the full dataset.

In [ ]:
import os, requests
import numpy as np
from tqdm.auto import tqdm
os.chdir('/content')

# Low-storage mode: do not download the 17.3 GB ACAV100M training array.
# Keep the official ~11-hour false-positive validation set (~185 MB).
import shutil
free_gb = shutil.disk_usage('/content').free / (1024**3)
print(f'Free storage before downloads: {free_gb:.1f} GB')
if free_gb < 4:
    raise RuntimeError(
        f'Not enough free storage ({free_gb:.1f} GB). Need at least 4 GB for models, '
        'generated clips, and training outputs.'
    )

# Download only the first 10% of ACAV100M using HTTP byte ranges (about 1.73 GB).
# This retains diverse real-world negative training data without storing the full 17.3 GB file.
import io
ACAV_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
TRAIN_FILE = '/content/acav_train_subset.npy'
ACAV_ROWS_FRACTION = 0.10
# A large file is not necessarily complete: validate its NumPy header and shape
# before trusting a cached download from a previous interrupted Colab session.
cache_ok = False
if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) > 1_500_000_000:
    try:
        cached = np.load(TRAIN_FILE, mmap_mode='r')
        cache_ok = (cached.ndim == 3 and cached.shape[1:] == (16, 96)
                    and cached.dtype == np.int16)
        if cache_ok:
            print(f'  cached: ACAV training subset {os.path.getsize(TRAIN_FILE)/1e9:.2f} GB; header and shape OK')
        del cached
    except Exception as e:
        print(f'  cached ACAV file is incomplete/invalid ({type(e).__name__}); will download again')
if not cache_ok:
    if os.path.exists(TRAIN_FILE):
        os.remove(TRAIN_FILE)
    with requests.get(ACAV_URL, headers={'Range': 'bytes=0-4095'}, timeout=120) as r:
        if r.status_code != 206:
            raise RuntimeError(f'ACAV server did not support HTTP range downloads (status {r.status_code}); stopping to avoid downloading 17.3 GB.')
        header_bytes = r.content
        bio = io.BytesIO(header_bytes)
        version = np.lib.format.read_magic(bio)
        shape, fortran_order, dtype = np.lib.format._read_array_header(bio, version)
        data_offset = bio.tell()
        if len(shape) != 3 or shape[1:] != (16, 96) or dtype != np.dtype('int16'):
            raise RuntimeError(f'Unexpected ACAV format: shape={shape}, dtype={dtype}.')
    n_rows = int(shape[0] * ACAV_ROWS_FRACTION)
    subset_bytes = n_rows * 16 * 96 * dtype.itemsize
    print(f'  source shape={shape}, dtype={dtype}; fetching {n_rows:,} rows (~{subset_bytes/1e9:.2f} GB, about 200 hours)')
    mm = np.lib.format.open_memmap(TRAIN_FILE, mode='w+', dtype=dtype, shape=(n_rows, 16, 96))
    raw = mm.view(np.uint8).reshape(-1)
    chunk_bytes = 64 * 1024 * 1024
    try:
        for start in tqdm(range(0, subset_bytes, chunk_bytes), desc='ACAV subset', unit='chunk'):
            end = min(start + chunk_bytes, subset_bytes) - 1
            remote_start = data_offset + start
            remote_end = data_offset + end
            with requests.get(ACAV_URL, headers={'Range': f'bytes={remote_start}-{remote_end}'}, stream=True, timeout=180) as r:
                if r.status_code != 206 or not r.headers.get('Content-Range', '').startswith(f'bytes {remote_start}-'):
                    raise RuntimeError('ACAV range response was not valid. No full-file fallback is attempted, to protect disk space.')
                block = bytearray()
                for piece in r.iter_content(chunk_size=4*1024*1024):
                    if piece: block.extend(piece)
                if len(block) != end - start + 1:
                    raise RuntimeError(f'Incomplete ACAV range: expected {end-start+1} bytes, got {len(block)}.')
                raw[start:end+1] = np.frombuffer(block, dtype=np.uint8)
        mm.flush()
    except Exception:
        del raw, mm
        if os.path.exists(TRAIN_FILE): os.remove(TRAIN_FILE)
        raise
    del raw, mm
    print(f'  saved ACAV training subset: {os.path.getsize(TRAIN_FILE)/1e9:.2f} GB')

VAL_FILE = '/content/validation_set_features.npy'
VAL_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'
if os.path.exists(VAL_FILE) and os.path.getsize(VAL_FILE) > 150_000_000:
    print(f'  cached: validation features {os.path.getsize(VAL_FILE)/1e6:.0f} MB')
else:
    part = VAL_FILE + '.part'
    resume_byte = os.path.getsize(part) if os.path.exists(part) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  downloading official false-positive validation set ({resume_byte/1e6:.0f} MB already cached)')
    with requests.get(VAL_URL, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        if resume_byte and r.status_code != 206:
            # Server ignored Range; restart safely rather than corrupt the file.
            resume_byte = 0
            mode = 'wb'
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(part, mode) as f, tqdm(total=total, initial=resume_byte,
                                         unit='B', unit_scale=True, unit_divisor=1024,
                                         desc='Validation features') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    os.replace(part, VAL_FILE)
    print(f'  saved validation features: {os.path.getsize(VAL_FILE)/1e6:.0f} MB')

# Stream a small background-music set instead of downloading the ~8 GB FMA archive.
import numpy as np
import scipy.io.wavfile as wavfile
from datasets import load_dataset, Audio
FMA_WAV_DIR = '/content/fma_wav'
os.makedirs(FMA_WAV_DIR, exist_ok=True)
existing_wavs = [f for f in os.listdir(FMA_WAV_DIR) if f.endswith('.wav')]
if len(existing_wavs) >= 120:
    print(f'  cached: {len(existing_wavs)} FMA background WAVs')
else:
    fma_stream = load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fma_stream = iter(fma_stream.cast_column('audio', Audio(sampling_rate=16000)))
    n_saved = 0
    for row in tqdm(fma_stream, total=120, desc='FMA stream (about 1 hour)'):
        audio = row['audio']
        arr = np.asarray(audio['array'], dtype=np.float32)
        if arr.ndim > 1:
            arr = arr.mean(axis=-1)
        out = f"{FMA_WAV_DIR}/{n_saved:05d}.wav"
        wavfile.write(out, 16000, (np.clip(arr, -1, 1) * 32767).astype(np.int16))
        n_saved += 1
        if n_saved >= 120:
            break
    if n_saved < 100:
        raise RuntimeError(f'FMA stream yielded only {n_saved} clips; need at least 100.')
    print(f'  saved {n_saved} background WAVs (about {n_saved/120:.1f} hour)')

## 8. Check streamed background WAVs

The previous cell writes up to 120 thirty-second background clips as 16-kHz mono WAVs. No full FMA archive is downloaded.

In [ ]:
import os
out_dir = '/content/fma_wav'
os.makedirs(out_dir, exist_ok=True)
wav_files = [f for f in os.listdir(out_dir) if f.endswith('.wav')]
if len(wav_files) >= 100:
    print(f'  cached: {len(wav_files)} streamed FMA WAVs ready')
else:
    raise RuntimeError(
        f'Only {len(wav_files)} background WAVs found in {out_dir}. '
        'Re-run the FMA streaming section in cell 14.'
    )


## 9. Prepare validation data

The official validation features are kept separate from training data. They are used to estimate false-positive activations across roughly 11 hours of representative audio.

In [ ]:
import os, numpy as np
TRAIN_FILE = '/content/acav_train_subset.npy'
VAL_FILE = '/content/validation_set_features.npy'
if not os.path.exists(TRAIN_FILE) or os.path.getsize(TRAIN_FILE) < 1_500_000_000:
    raise RuntimeError('ACAV training subset is missing or incomplete; re-run cell 7.')
train = np.load(TRAIN_FILE, mmap_mode='r')
print(f'  ACAV training subset: shape={train.shape}, dtype={train.dtype}, size={os.path.getsize(TRAIN_FILE)/1e9:.2f} GB')
if train.ndim != 3 or train.shape[1:] != (16, 96) or train.dtype != np.int16:
    raise RuntimeError(f'Unexpected ACAV training subset format: shape={train.shape}, dtype={train.dtype}.')
if not os.path.exists(VAL_FILE) or os.path.getsize(VAL_FILE) < 150_000_000:
    raise RuntimeError('Official validation_set_features.npy is missing or incomplete; re-run cell 7.')
val = np.load(VAL_FILE, mmap_mode='r')
print(f'  official validation features: shape={val.shape}, dtype={val.dtype}')
if val.ndim != 2 or val.shape[-1] != 96:
    raise RuntimeError(f'Unexpected validation feature shape {val.shape}; expected (frames, 96).')
print('  training and validation feature files passed structural checks.')

## 10. Build training config (no example-yaml dependency)

Self-contained â€” every field set explicitly. Doesn't depend on upstream's `examples/` directory which has been renamed/removed across openwakeword versions.

In [ ]:
import yaml, os

# ╔═══════════════════════════════════════════════════════════════════╗
# ║                   ★  EDIT THESE TWO LINES  ★                      ║
# ║                                                                   ║
# ║  TARGET_PHRASE: list of strings — what the model wakes on.        ║
# ║                 Multiple variants OK (e.g. "mr graves" plus       ║
# ║                 "mister graves") — they all map to the same       ║
# ║                 binary classifier output.                         ║
# ║                                                                   ║
# ║  MODEL_NAME:    short string — used for output filename + dirs.   ║
# ║                 No spaces, lowercase, snake_case.                 ║
# ╚═══════════════════════════════════════════════════════════════════╝
TARGET_PHRASE = ['falkor', 'falcor', 'fal core', 'fal kor']
MODEL_NAME    = 'falkor'
# ── (you can leave everything else below alone) ──────────────────────

# Self-contained config — every field upstream train.py reads, set explicitly.
# Doesn't depend on any example yaml from the openwakeword repo (which has been
# renamed/removed across versions). Keys derived from grepping
# `config["..."]` in upstream train.py @ master 2026-05-09.
config = {
    # ── Wake-word ───────────────────────────────────────────────────────
    'target_phrase':   TARGET_PHRASE,
    'model_name':      MODEL_NAME,
    # Empty list → only auto-generated adversarial phonemes used as negatives.
    # Upstream code (line 707/730) does:
    #   adversarial_texts = config["custom_negative_phrases"]
    #   adversarial_texts.extend(generate_adversarial_texts(...))
    # — so the key MUST exist as a mutable list (KeyError otherwise) but
    # `[]` is fine; auto-generated negatives still get appended.
    'custom_negative_phrases': [],

    # ── Synthetic clip generation ───────────────────────────────────────
    'n_samples':       2000,    # bump for harder phrases / longer training
    'n_samples_val':   1000,
    'tts_batch_size':  50,
    'piper_sample_generator_path': '/content/piper-sample-generator',

    # ── Augmentation ────────────────────────────────────────────────────
    'augmentation_rounds':    1,
    # Upstream comment in custom_model.yml: "not too large to ensure variety
    # in augmentation". Upstream default = 16.
    'augmentation_batch_size': 16,

    # ── Training (read by hand-rolled trainer in cell 14) ──────────────
    'steps':           20000,
    'max_negative_weight': 1500,    # bump to 3000 if FP rate too high after stage 1
    'target_accuracy': 0.7,
    'target_recall':   0.5,
    'target_false_positives_per_hour': 0.5,
    'batch_size':      128,
    'learning_rate':   1e-4,

    # ── Model arch (matches openwakeword's DNN exactly) ────────────────
    'model_type':      'dnn',
    'layer_dim':       128,
    # Upstream's --train_model path (line 826) reads `layer_size` not layer_dim.
    'layer_size':      128,
    'n_blocks':        1,
    'model_input_shape': [16, 96],
    'n_classes':       1,
    'batch_n_per_class': {
        'ACAV100M_sample':       64,
        'adversarial_negative':    50,
        'positive':                50,
    },

    # ── Data paths ──────────────────────────────────────────────────────
    'background_paths': ['/content/fma_wav'],
    'background_paths_duplication_rate': [1],
    'rir_paths':        ['/content/mit_rirs'],
    'false_positive_validation_data_path': '/content/validation_set_features.npy',
    'feature_data_files': {'ACAV100M_sample': '/content/acav_train_subset.npy'},
    'output_dir':       f'/content/{MODEL_NAME}_output',
    'tflite_export':    False,
    'onnx_export':      True,

    # ── Clip dirs (used by hand-rolled trainer) ────────────────────────
    'positive_clips_train_dir': f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_train',
    'positive_clips_test_dir':  f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_test',
    'negative_clips_train_dir': f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_train',
    'negative_clips_test_dir':  f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_test',
    'feature_save_dir':         f'/content/{MODEL_NAME}_output/{MODEL_NAME}',
}
os.makedirs(config['output_dir'], exist_ok=True)
with open('/content/my_model.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print(f'  target_phrase: {config["target_phrase"]}')
print(f'  model_name:    {config["model_name"]}')
print(f'  steps:         {config["steps"]}')
print(f'  max_neg_w:     {config["max_negative_weight"]}')
print(f'  target_recall: {config["target_recall"]}')
print(f'  target_fp/hr:  {config["target_false_positives_per_hour"]}')

## 11. Generate Piper TTS clips (~10-15 min)

Uses openwakeword's --generate_clips runner which reads from my_model.yaml. Idempotent â€” re-running finds existing clips and skips.

In [ ]:
import sys, os, yaml
# Idempotent: check ALL FOUR clip dirs and skip the generate_clips invocation
# only if every dir is fully populated. Upstream's `n_current_samples <= 0.95
# *n_samples` checks at lines 675/692/706/729 in train.py also skip cached
# dirs internally, so re-running is safe even if some dirs are partial.
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
dirs = {
    'positive_train': (cfg['positive_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'positive_test':  (cfg['positive_clips_test_dir'],  int(cfg['n_samples_val'] * 0.75)),
    'negative_train': (cfg['negative_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'negative_test':  (cfg['negative_clips_test_dir'],  int(cfg['n_samples_val'] * 0.75)),
}
status = []
for name, (path, expected) in dirs.items():
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    status.append((name, n, expected, n >= expected))
all_full = all(ok for _, _, _, ok in status)
for name, n, expected, ok in status:
    print(f'  {"PASS" if ok else "MISS"} {name}: {n} clips (expect >={expected})')
if all_full:
    print('\n  cached: all 4 clip dirs already populated')
else:
    print('\n  -> running generate_clips (upstream skips cached dirs internally)')
    !{sys.executable} /content/openwakeword/openwakeword/train.py \
        --training_config /content/my_model.yaml \
        --generate_clips
    for name, (path, expected) in dirs.items():
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        assert n >= expected, f'generate_clips left {name} with only {n} clips (expected >={expected})'
    print('\n  PASS all 4 clip dirs now populated')

## 12. Resample TTS clips 22050 â†’ 16000 Hz

Piper TTS outputs at libritts's native 22050 Hz. The augment + feature pipelines expect 16 kHz.

In [ ]:
import os, glob, soundfile as sf, yaml
from scipy.signal import resample_poly
from tqdm.auto import tqdm
TARGET_SR = 16000
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
ROOT = cfg['output_dir']
wav_dirs = sorted({os.path.dirname(f)
                   for f in glob.glob(f'{ROOT}/**/*.wav', recursive=True)})
for d in wav_dirs:
    files = [f for f in os.listdir(d) if f.endswith('.wav')]
    if not files: continue
    sr_counts = {}
    for f in files[:20]:
        sr = sf.info(f'{d}/{f}').samplerate
        sr_counts[sr] = sr_counts.get(sr, 0) + 1
    if all(k == TARGET_SR for k in sr_counts):
        print(f'  ok {d}: {len(files)} files at 16 kHz (skip)')
        continue
    print(f'  resampling {d}: {len(files)} files, SRs {sr_counts}')
    n = 0
    for f in tqdm(files, desc=os.path.basename(d) or d, leave=False):
        p = f'{d}/{f}'
        data, sr = sf.read(p)
        if sr == TARGET_SR: continue
        new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
        sf.write(p, new_data, TARGET_SR); n += 1
    print(f'  ok {d}: resampled {n}/{len(files)}')
# Clear stale .npy features so augment re-runs cleanly
for f in glob.glob(f'{ROOT}/**/*.npy', recursive=True):
    os.remove(f)
print('\n  cleared stale features (will be regenerated by augment)')

## 13. Augment + featurise (~10 min)

Outputs `(N, 16, 96)` feature .npy files that the hand-rolled trainer reads.

In [ ]:
import sys, os, glob, yaml
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
FEAT = cfg['feature_save_dir']
needed = ['positive_features_train.npy', 'negative_features_train.npy',
          'positive_features_test.npy',  'negative_features_test.npy']
if all(os.path.exists(f'{FEAT}/{n}') for n in needed):
    print('  cached: all 4 feature .npy files exist')
else:
    !{sys.executable} /content/openwakeword/openwakeword/train.py \
        --training_config /content/my_model.yaml \
        --augment_clips
for f in sorted(glob.glob(f'{FEAT}/*.npy')):
    print(f'  {f}: {os.path.getsize(f)/1e6:.1f} MB')
for n in needed:
    assert os.path.exists(f'{FEAT}/{n}'), f"augment didn't produce {n}"
print('  PASS all 4 feature files present')

## 14. Hand-rolled trainer (~30-40 min on L4 / A100)

Mirrors openWakeWord's `auto_train`: 3-stage lr schedule (1e-4 â†’ 1e-5 â†’ 1e-6), neg_weight ramp 1â†’1500, hard-negative mining, FP/hour validation against ACAV val, percentile-based checkpoint saves.

In [ ]:
import os, copy, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
FEAT = cfg['feature_save_dir']
pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
print(f'  pos_train={tuple(pos_train.shape)}  neg_train={tuple(neg_train.shape)}')
print(f'  pos_test={tuple(pos_test.shape)}  neg_test={tuple(neg_test.shape)}')

acav_train_np = np.load(cfg['feature_data_files']['ACAV100M_sample'], mmap_mode='r')
print(f'  acav_train (mmap): {acav_train_np.shape}')
acav_val_np = np.load(cfg['false_positive_validation_data_path'])
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
print(f'  acav_val: {acav_val_np.shape}  â†’ {val_listen_hours:.2f} hr listening time')
# sliding_window_view is a view: do not materialize the full overlapping
# validation windows as float32, which can consume several GB of RAM.
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
n_win_val = acav_val_windows.shape[0]
print(f'  acav_val_windows: {acav_val_windows.shape} (view; converts one batch at a time)')
VAL_BATCH = 4096

class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

model = WakewordModel().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
TOTAL_STEPS = cfg['steps']
MAX_NEG_W   = cfg['max_negative_weight']
TARGET_FP_PER_HR = cfg['target_false_positives_per_hour']
THRESH = 0.5
history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
best_models = []

B_POS, B_ANEG, B_ACAV = 32, 32, 64
def random_acav_window_batch(k):
    N, T, F = acav_train_np.shape
    rows = np.random.randint(0, N, size=k)
    starts = np.random.randint(0, T - 16 + 1, size=k)
    out = np.empty((k, 16, F), dtype=np.float32)
    for i, (r, s) in enumerate(zip(rows, starts)):
        out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
    return out

def build_batch():
    p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
    aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
    p, an = pos_train[p_idx], neg_train[aneg_idx]
    acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
    x = torch.cat([p, an, acav], dim=0)
    y = torch.cat([torch.ones(B_POS, device=DEVICE),
                   torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
    return x, y

@torch.no_grad()
def validate(step_label):
    model.eval()
    p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
    n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
    recall = (p_preds >= THRESH).float().mean().item()
    accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                / (pos_test.shape[0] + neg_test.shape[0]))
    n_fp = 0
    for i in range(0, n_win_val, VAL_BATCH):
        chunk_np = acav_val_windows[i:i+VAL_BATCH].astype(np.float32, copy=True)
        chunk = torch.from_numpy(chunk_np).to(DEVICE)
        n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
    fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
    history['val_recall'].append(recall)
    history['val_accuracy'].append(accuracy)
    history['val_fp_per_hour'].append(fp_per_hour)
    history['val_n_fp'].append(n_fp)
    save = False
    if len(history['val_n_fp']) >= 3:
        fp_p50 = np.percentile(history['val_n_fp'], 50)
        rc_p5  = np.percentile(history['val_recall'], 5)
        if n_fp <= fp_p50 and recall >= rc_p5:
            best_models.append((copy.deepcopy(model.state_dict()),
                                {'val_recall': recall, 'val_accuracy': accuracy,
                                 'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
            save = True
    print(f'  [{step_label}] recall={recall:.3f}  acc={accuracy:.3f}  fp/hr={fp_per_hour:.2f}  saved={"+" if save else "-"}', flush=True)
    model.train()

def run_stage(stage_idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
    val_start = int(n_steps * (1.0 - val_window_frac))
    val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
    warmup = max(1, n_steps // 5)
    hold   = n_steps // 3
    accumulated = []
    print(f'\n=== Stage {stage_idx}: {n_steps} steps, lr={lr}, max_neg_w={max_neg_w} ===', flush=True)
    t0 = time.time()
    for step in range(n_steps):
        if step < warmup:
            lr_now = lr * (step + 1) / warmup
        elif step < warmup + hold:
            lr_now = lr
        else:
            decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
            lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
        for pg in optimizer.param_groups: pg['lr'] = lr_now
        x, y = build_batch()
        logits = model(x).squeeze(-1)
        preds = torch.sigmoid(logits)
        keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
        if keep.sum() == 0:
            if step in val_steps: validate(f'stage{stage_idx} step {step}/{n_steps}')
            continue
        kept_logits = logits[keep]
        kept_y = y[keep]
        neg_w = weight_schedule[step]
        w = torch.where(kept_y > 0.5,
                        torch.tensor(1.0, device=DEVICE),
                        torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
        accumulated.append((kept_logits, kept_y, w))
        if sum(t[0].shape[0] for t in accumulated) >= 128:
            cat_logits = torch.cat([t[0] for t in accumulated])
            cat_y = torch.cat([t[1] for t in accumulated])
            cat_w = torch.cat([t[2] for t in accumulated])
            loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            history['loss'].append(loss.item())
            accumulated.clear()
        if step in val_steps:
            elapsed = (time.time() - t0) / 60
            validate(f'stage{stage_idx} step {step}/{n_steps} ({elapsed:.1f}m, lr={lr_now:.6f}, neg_w={neg_w:.0f})')
    print(f'  Stage {stage_idx} done in {(time.time()-t0)/60:.1f} min', flush=True)

stage1_steps = TOTAL_STEPS
stage2_steps = max(2000, TOTAL_STEPS // 10)
stage3_steps = max(2000, TOTAL_STEPS // 10)
max_neg_w_now = MAX_NEG_W
run_stage(1, stage1_steps, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage1 best FP/hr > target â†’ max_neg_w doubled to {max_neg_w_now}')
run_stage(2, stage2_steps, lr=1e-5, max_neg_w=max_neg_w_now, val_window_frac=1.0)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage2 best FP/hr > target â†’ max_neg_w doubled to {max_neg_w_now}')
run_stage(3, stage3_steps, lr=1e-6, max_neg_w=max_neg_w_now, val_window_frac=1.0)
print(f'\n=== Training done. {len(best_models)} checkpoints saved. ===')
print(f'  best val_fp_per_hour: {min(history["val_fp_per_hour"]):.2f}')
print(f'  best val_recall:      {max(history["val_recall"]):.3f}')
print(f'  best val_accuracy:    {max(history["val_accuracy"]):.3f}')

## 15. Ensemble best checkpoints + ONNX export + download

90/90/10 percentile filter on saved checkpoints, average state_dicts, export ONNX with sigmoid baked in, sanity-check, browser-download.

In [ ]:
import copy, os
import numpy as np
import torch
if not best_models:
    print('  no saved checkpoints â€” using current model state')
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s['val_accuracy'] for _, s in best_models]
    rcs  = [s['val_recall']   for _, s in best_models]
    fps  = [s['val_fp_per_hour'] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90  = np.percentile(rcs, 90)
    fp_p10  = np.percentile(fps, 10)
    print(f'  thresholds: accâ‰¥{acc_p90:.3f}  recallâ‰¥{rc_p90:.3f}  fp/hrâ‰¤{fp_p10:.2f}')
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                 and sc['val_fp_per_hour'] <= fp_p10]
    print(f'  qualified: {len(qualified)} of {len(best_models)} checkpoints')
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))
        qualified = [sorted_models[0]]
        print('  no checkpoint qualified â€” fell back to single best')
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    qa = np.mean([s['val_accuracy'] for _, s in qualified])
    qr = np.mean([s['val_recall'] for _, s in qualified])
    qf = np.mean([s['val_fp_per_hour'] for _, s in qualified])
    print(f'  ensemble averaged: mean acc={qa:.3f}, recall={qr:.3f}, fp/hr={qf:.2f}')
model.load_state_dict(final_state)
model.eval()

# Export with sigmoid baked in so APK runtime sees [0, 1]
out_path = f"{cfg['feature_save_dir']}/{cfg['model_name']}.onnx"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
class WakewordExportable(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x): return torch.sigmoid(self.base(x))
export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)
torch.onnx.export(export_model, dummy, out_path,
                  input_names=['onnx::Flatten_0'], output_names=['output'],
                  dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                  opset_version=14, dynamo=False)
print(f'  âœ“ wrote {out_path} ({os.path.getsize(out_path)/1e3:.0f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(out_path)
print(f'  input:  {sess.get_inputs()[0].name} {sess.get_inputs()[0].shape}')
print(f'  output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}')
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    p_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f'  positive test set: mean={p_scores.mean():.3f}, recall@0.5={(p_scores>=0.5).mean():.3f}')
print(f'  ACAV val FP/hour at 0.5: {history["val_fp_per_hour"][-1]:.2f}')

from google.colab import files
files.download(out_path)
print(f'\nDONE. Wake word "{cfg["model_name"]}" trained.')
print('Drop the downloaded .onnx into your APK / on-device app at:')
print(f'  android/app/src/main/assets/wakewords/{cfg["model_name"]}.onnx  (or wherever your APK looks)')